<a href="https://colab.research.google.com/github/JoshuaFZ/QWEN-0.6B-LORA/blob/main/rkllm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RKLLM Colab GPU conversion

This notebook converts a HuggingFace model to RKLLM on Google Colab with GPU.

Before running it, choose Runtime, Change runtime type, GPU in Colab.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configure paths and build options

Change these paths to match your Google Drive layout. Put the rkllm-toolkit wheel files from rkllm-toolkit/packages into Drive, or set RKLLM_TOOLKIT_WHEEL directly.


In [ ]:
from pathlib import Path
from datetime import datetime

MODEL_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output-mergeable-bf16/owon-qwen3-0.6b-merged-peft-safe-bf16')
RKLLM_TOOLKIT_PACKAGE_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/rkllm-toolkit/packages')
RKLLM_TOOLKIT_WHEEL = None
RKLLM_TOOLKIT_VERSION = '1.2.2'
ALLOW_RKNN_LLM_CLONE = False

DATA_QUANT = MODEL_DIR / 'data_quant.json'
JSONL_DATASET = Path('/content/drive/MyDrive/qwen3-0.6B/LORA_train-qwen0.6B.jsonl')
MAX_CALIBRATION_ROWS = None
REGENERATE_DATA_QUANT = True

OUTPUT_DIR = MODEL_DIR
OUTPUT_BASENAME = 'owon-qwen3-0.6b-mergeable-bf16-peft-safe'
TARGET_PLATFORM = 'RK3588'
QUANTIZED_DTYPE = 'W8A8'
QUANTIZED_ALGORITHM = 'normal'
OPTIMIZATION_LEVEL = 0
NUM_NPU_CORE = 3
MAX_CONTEXT = 4096
LOAD_DTYPE = 'float32'

DATE_TAG = datetime.now().strftime('%Y%m%d')
OUTPUT_PATH = OUTPUT_DIR / f'{OUTPUT_BASENAME}_{QUANTIZED_DTYPE}_{TARGET_PLATFORM}_{DATE_TAG}.rkllm'
print('MODEL_DIR =', MODEL_DIR)
print('DATA_QUANT =', DATA_QUANT)
print('OUTPUT_PATH =', OUTPUT_PATH)



## 3b. Install rkllm-toolkit wheel and verify CUDA

Run this after the runtime restarts from step 3a. The cell searches Drive, /content, and a cloned RKNN-LLM repo for the matching rkllm-toolkit wheel. The wheel is installed with --no-deps because its metadata includes optional/problematic packages such as auto_gptq.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import importlib.metadata as importlib_metadata
import importlib.util

REPO_URL = 'https://github.com/airockchip/rknn-llm.git'
REPO_DIR = Path('/content/rknn-llm')

# Colab runtime restart clears variables from previous cells. Provide safe defaults.
RKLLM_TOOLKIT_VERSION = globals().get('RKLLM_TOOLKIT_VERSION', '1.2.2')
ALLOW_RKNN_LLM_CLONE = globals().get('ALLOW_RKNN_LLM_CLONE', False)
RKLLM_TOOLKIT_WHEEL = globals().get('RKLLM_TOOLKIT_WHEEL', None)
RKLLM_TOOLKIT_PACKAGE_DIR = Path(globals().get(
    'RKLLM_TOOLKIT_PACKAGE_DIR',
    '/content/drive/MyDrive/qwen3-0.6B/rkllm-toolkit/packages',
))
py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
RKLLM_MARKER = Path(f'/content/.rkllm_toolkit_installed_{RKLLM_TOOLKIT_VERSION}_{py_tag}')


def run(cmd):
    print('+', ' '.join(map(str, cmd)), flush=True)
    subprocess.check_call(list(map(str, cmd)))


def installed_rkllm_toolkit_version():
    try:
        return importlib_metadata.version('rkllm_toolkit')
    except importlib_metadata.PackageNotFoundError:
        return None


def find_rkllm_wheel():
    wheel_pattern = f'rkllm_toolkit-{RKLLM_TOOLKIT_VERSION}-{py_tag}-{py_tag}-linux_x86_64.whl'

    wheel = RKLLM_TOOLKIT_WHEEL
    if wheel is not None:
        wheel = Path(wheel)
        if wheel.exists():
            if wheel.name != wheel_pattern:
                raise RuntimeError(f'RKLLM_TOOLKIT_WHEEL must be {wheel_pattern}, got: {wheel.name}')
            return wheel

    candidates = []
    # Do not recursively scan /content: it contains /content/drive and can traverse all of MyDrive.
    if RKLLM_TOOLKIT_PACKAGE_DIR.exists():
        candidates.extend(sorted(RKLLM_TOOLKIT_PACKAGE_DIR.glob(wheel_pattern)))
    candidates.extend(sorted(Path('/content').glob(wheel_pattern)))
    if candidates:
        return candidates[-1]

    if ALLOW_RKNN_LLM_CLONE:
        print('No local rkllm-toolkit 1.2.2 wheel found. Cloning RKNN-LLM repo to /content...', flush=True)
        if not REPO_DIR.exists():
            run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
        package_dir = REPO_DIR / 'rkllm-toolkit' / 'packages'
        if package_dir.exists():
            candidates = sorted(package_dir.glob(wheel_pattern))
            if candidates:
                return candidates[-1]
    else:
        print('Skip RKNN-LLM git clone. Set ALLOW_RKNN_LLM_CLONE=True if you want to search a freshly cloned repo.')

    print('Still no matching wheel found after exact package-dir and /content checks.')
    print('Current Python requires:', wheel_pattern)
    print('You can upload the matching wheel now. For this repo, use:')
    print('rkllm-toolkit/packages/' + wheel_pattern)
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_candidates = [Path(name) for name in uploaded if Path(name).name == wheel_pattern]
        if uploaded_candidates:
            return uploaded_candidates[-1]
    except Exception as exc:
        print('Upload helper is unavailable:', exc)

    raise FileNotFoundError(
        'No matching rkllm-toolkit wheel was found.\n'
        f'Current Python requires: {wheel_pattern}\n'
        f'Default expected directory: {RKLLM_TOOLKIT_PACKAGE_DIR}\n'
        'Copy/upload the 1.2.2 wheel, or set RKLLM_TOOLKIT_WHEEL to the exact .whl path.'
    )


current_version = installed_rkllm_toolkit_version()
if current_version != RKLLM_TOOLKIT_VERSION or not RKLLM_MARKER.exists():
    wheel = find_rkllm_wheel()
    print('Using rkllm-toolkit wheel:', wheel)
    if current_version is not None:
        print('Existing rkllm-toolkit version:', current_version, '-> reinstalling', RKLLM_TOOLKIT_VERSION)
    run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', str(wheel)])
    RKLLM_MARKER.write_text(str(wheel), encoding='utf-8')
else:
    print('RKLLM toolkit marker found:', RKLLM_MARKER)
    print('Installed wheel:', RKLLM_MARKER.read_text(encoding='utf-8').strip())

BACKEND_CLEAN_MARKER = Path(f'/content/.rkllm_tf_jax_removed_{py_tag}')
if not BACKEND_CLEAN_MARKER.exists():
    print('Removing preinstalled TF/JAX/Flax backends...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'tensorflow', 'tensorflow-cpu', 'tensorflow-probability', 'tensorflow-datasets', 'keras', 'jax', 'jaxlib', 'flax'], check=False)
    BACKEND_CLEAN_MARKER.write_text('ok', encoding='utf-8')
    print('Backend cleanup done. Restarting runtime to clear import caches...', flush=True)
    print('After Colab reconnects, rerun this same 3b cell.', flush=True)
    os.kill(os.getpid(), 9)
print('Backend cleanup marker found:', BACKEND_CLEAN_MARKER)

# Install RKLLM runtime dependencies without touching torch/transformers/numpy.
# Use this path when you want to preserve the LoRA/HF validation environment
# (for example torch 2.11 + transformers 5.5) while still importing rkllm-toolkit.
RKLLM_RUNTIME_DEPS_NO_TORCH_TRANSFORMERS = [
    'colorlog==6.8.2',
    'tabulate==0.9.0',
    'jsonschema==4.23.0',
    'flatbuffers==24.3.25',
    'jsonlines==4.0.0',
    'sentencepiece==0.2.0',
    'protobuf<=4.25.4,>=4.21.6',
    'transformers_stream_generator==0.0.5',
    'einops==0.4.1',
    'tiktoken<=0.9.0,>=0.7.0',
    'datamodel_code_generator==0.26.0',
    'optimum==1.23.3',
    'timm==1.0.19',
]
RKLLM_RUNTIME_IMPORT_CHECKS = [
    'colorlog',
    'tabulate',
    'jsonschema',
    'flatbuffers',
    'jsonlines',
    'sentencepiece',
    'google.protobuf',
    'transformers_stream_generator',
    'einops',
    'tiktoken',
    'datamodel_code_generator',
    'optimum',
    'timm',
]

def missing_rkllm_runtime_modules():
    missing = []
    for module_name in RKLLM_RUNTIME_IMPORT_CHECKS:
        try:
            if importlib.util.find_spec(module_name) is None:
                missing.append(module_name)
        except ModuleNotFoundError:
            missing.append(module_name)
    return missing

RKLLM_RUNTIME_DEPS_MARKER = Path(f'/content/.rkllm_runtime_deps_no_torch_transformers_{py_tag}')
missing_runtime_modules = missing_rkllm_runtime_modules()
if (not RKLLM_RUNTIME_DEPS_MARKER.exists()) or missing_runtime_modules:
    if missing_runtime_modules:
        print('Missing RKLLM runtime modules:', missing_runtime_modules)
    print('Installing RKLLM runtime deps without changing torch/transformers/numpy...', flush=True)
    run([sys.executable, '-m', 'pip', 'install', '-U', *RKLLM_RUNTIME_DEPS_NO_TORCH_TRANSFORMERS])
    missing_runtime_modules = missing_rkllm_runtime_modules()
    if missing_runtime_modules:
        raise ModuleNotFoundError(f'RKLLM runtime deps still missing after install: {missing_runtime_modules}')
    RKLLM_RUNTIME_DEPS_MARKER.write_text('ok', encoding='utf-8')
else:
    print('RKLLM runtime deps marker found and modules verified:', RKLLM_RUNTIME_DEPS_MARKER)

# Disable TensorFlow/Flax backends in transformers. RKLLM conversion only needs PyTorch.
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['USE_JAX'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
for module_name in list(sys.modules):
    if module_name == 'rkllm' or module_name.startswith('rkllm.'):
        sys.modules.pop(module_name, None)
    if module_name == 'transformers' or module_name.startswith(('transformers.', 'tensorflow', 'jax', 'flax')):
        sys.modules.pop(module_name, None)

import numpy as np
import torch
print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. In Colab, select Runtime, Change runtime type, GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from rkllm.api import RKLLM
installed_version = installed_rkllm_toolkit_version()
print('RKLLM toolkit import OK, version:', installed_version)
if installed_version != RKLLM_TOOLKIT_VERSION:
    raise RuntimeError(f'Expected rkllm-toolkit {RKLLM_TOOLKIT_VERSION}, got {installed_version}')

## 4.Prepare quantization data

If DATA_QUANT already exists, it is reused. Otherwise the notebook generates it from JSONL_DATASET using an Alpaca-style prompt.


In [ ]:
import json
# Restore configuration after Colab runtime restart.
from pathlib import Path
from datetime import datetime

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

if 'MODEL_DIR' not in globals():
    MODEL_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output-mergeable-bf16/owon-qwen3-0.6b-merged-peft-safe-bf16')
if 'RKLLM_TOOLKIT_PACKAGE_DIR' not in globals():
    RKLLM_TOOLKIT_PACKAGE_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/rkllm-toolkit/packages')
if 'RKLLM_TOOLKIT_WHEEL' not in globals():
    RKLLM_TOOLKIT_WHEEL = None
if 'RKLLM_TOOLKIT_VERSION' not in globals():
    RKLLM_TOOLKIT_VERSION = '1.2.2'
if 'ALLOW_RKNN_LLM_CLONE' not in globals():
    ALLOW_RKNN_LLM_CLONE = False
if 'DATA_QUANT' not in globals():
    DATA_QUANT = MODEL_DIR / 'data_quant.json'
if 'JSONL_DATASET' not in globals():
    JSONL_DATASET = Path('/content/drive/MyDrive/qwen3-0.6B/LORA_train-qwen0.6B.jsonl')
if 'REGENERATE_DATA_QUANT' not in globals():
    REGENERATE_DATA_QUANT = True
if 'MAX_CALIBRATION_ROWS' not in globals():
    MAX_CALIBRATION_ROWS = None
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = MODEL_DIR
if 'OUTPUT_BASENAME' not in globals():
    OUTPUT_BASENAME = 'owon-qwen3-0.6b-mergeable-bf16-peft-safe'
if 'TARGET_PLATFORM' not in globals():
    TARGET_PLATFORM = 'RK3588'
if 'QUANTIZED_DTYPE' not in globals():
    QUANTIZED_DTYPE = 'W8A8'
if 'QUANTIZED_ALGORITHM' not in globals():
    QUANTIZED_ALGORITHM = 'normal'
if 'OPTIMIZATION_LEVEL' not in globals():
    OPTIMIZATION_LEVEL = 0
if 'NUM_NPU_CORE' not in globals():
    NUM_NPU_CORE = 3
if 'MAX_CONTEXT' not in globals():
    MAX_CONTEXT = 4096
if 'LOAD_DTYPE' not in globals():
    LOAD_DTYPE = 'float32'
if 'OUTPUT_PATH' not in globals():
    DATE_TAG = datetime.now().strftime('%Y%m%d')
    OUTPUT_PATH = OUTPUT_DIR / f'{OUTPUT_BASENAME}_{QUANTIZED_DTYPE}_{TARGET_PLATFORM}_{DATE_TAG}.rkllm'
print('MODEL_DIR =', MODEL_DIR)
print('DATA_QUANT =', DATA_QUANT)
print('OUTPUT_PATH =', OUTPUT_PATH)


# Keep quantization calibration in the same distribution as LoRA training and HF eval.
SCHEMA_INSTRUCTION = (
    '你是一个示波器语音助手，请解析用户语音文本，仅输出一个紧凑JSON对象。字段固定为intent和slots；'
    '缺少必选槽位时增加missing_slots数组；暂未支持或需求待确认时增加unsupported或needs_confirm。'
    '不要输出解释，不要输出Markdown，不要直接输出SCPI。'
)

PROMPT_TEMPLATE = '''任务：解析示波器语音指令，只输出JSON。
规则：{}
输入：{}
输出：{}'''

def build_data_quant(jsonl_path, output_path, limit=None):
    rows = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            target = record.get('output', '')
            if not isinstance(target, str):
                target = json.dumps(target, ensure_ascii=False, separators=(',', ':'))
            rows.append({
                'input': PROMPT_TEMPLATE.format(SCHEMA_INSTRUCTION, record.get('input', ''), ''),
                'target': target,
            })
            if limit and len(rows) >= limit:
                break
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    return len(rows)

if not DATA_QUANT.exists():
    quant_candidates = [
        MODEL_DIR / 'data_quant.json',
        Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output-mergeable-bf16/owon-qwen3-0.6b-merged-peft-safe-bf16/data_quant.json'),
    ]
    quant_candidates = [p for p in quant_candidates if p.exists()]
    if quant_candidates:
        DATA_QUANT = quant_candidates[0]
        print('Using discovered quant data:', DATA_QUANT)
if not DATA_QUANT.exists() and not JSONL_DATASET.exists():
    jsonl_candidates = [
        Path('/content/drive/MyDrive/qwen3-0.6B/LORA_train-qwen0.6B.jsonl'),
        MODEL_DIR.parent.parent / 'LORA_train-qwen0.6B.jsonl',
    ]
    jsonl_candidates = [p for p in jsonl_candidates if p.exists()]
    if jsonl_candidates:
        JSONL_DATASET = jsonl_candidates[0]
        print('Using JSONL dataset:', JSONL_DATASET)


if REGENERATE_DATA_QUANT and JSONL_DATASET.exists():
    n = build_data_quant(JSONL_DATASET, DATA_QUANT, MAX_CALIBRATION_ROWS)
    print(f'Regenerated quant data: {DATA_QUANT} ({n} records)')
elif DATA_QUANT.exists():
    with open(DATA_QUANT, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    print(f'Using existing quant data: {DATA_QUANT} ({len(rows)} records)')
else:
    raise FileNotFoundError(f'DATA_QUANT not found and JSONL_DATASET does not exist: {JSONL_DATASET}')

with open(DATA_QUANT, 'r', encoding='utf-8') as f:
    preview_rows = json.load(f)[:1]
if preview_rows:
    print('--- quant prompt preview ---')
    print(preview_rows[0]['input'])
    print('target:', preview_rows[0].get('target'))




## 4b. Clean native Transformers validation before RKLLM conversion

Set `RUN_MERGED_HF_EVAL = True` to validate the saved HuggingFace merged directory in a clean Python subprocess before RKLLM conversion. The subprocess imports only Transformers, not Unsloth, so the result is the real conversion-time HF baseline.

If this score is close to the LoRA notebook safe-merged baseline but RKLLM is low, continue with RKLLM quantization/conversion diagnosis. If this score is low, fix the saved HF model/config/tokenizer first.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

RUN_MERGED_HF_EVAL = globals().get('RUN_MERGED_HF_EVAL', False)
if 'HF_EVAL_JSONL' in globals():
    HF_EVAL_JSONL = Path(HF_EVAL_JSONL)
else:
    hf_eval_candidates = [
        Path('/content/drive/MyDrive/VoiceControl2/training/qwen3-0.6B/LORA_generalization_test-qwen0.6B.jsonl'),
        Path('/content/drive/MyDrive/qwen3-0.6B/LORA_generalization_test-qwen0.6B.jsonl'),
        Path('/content/drive/MyDrive/LORA_generalization_test-qwen0.6B.jsonl'),
        MODEL_DIR.parent.parent / 'LORA_generalization_test-qwen0.6B.jsonl',
    ]
    HF_EVAL_JSONL = next((p for p in hf_eval_candidates if p.exists()), hf_eval_candidates[0])
MAX_HF_EVAL_ROWS = globals().get('MAX_HF_EVAL_ROWS', None)
HF_EVAL_MAX_NEW_TOKENS = globals().get('HF_EVAL_MAX_NEW_TOKENS', 96)
HF_EVAL_DTYPES = globals().get('HF_EVAL_DTYPES', ['bfloat16'])

RUN_MERGED_HF_EVAL = True
if not RUN_MERGED_HF_EVAL:
    print('Skip clean merged HF eval. Set RUN_MERGED_HF_EVAL=True and rerun this cell before RKLLM conversion.')
else:
    if not Path(MODEL_DIR).exists():
        raise FileNotFoundError(f'MODEL_DIR not found: {MODEL_DIR}')
    if not HF_EVAL_JSONL.exists():
        raise FileNotFoundError(f'HF_EVAL_JSONL not found: {HF_EVAL_JSONL}')

    script_path = Path('/tmp/clean_transformers_eval_before_rkllm.py')
    config_path = Path('/tmp/clean_transformers_eval_before_rkllm_config.json')
    config = {
        'model_dir': str(MODEL_DIR),
        'jsonl': str(HF_EVAL_JSONL),
        'dtypes': HF_EVAL_DTYPES,
        'max_rows': MAX_HF_EVAL_ROWS,
        'max_new_tokens': HF_EVAL_MAX_NEW_TOKENS,
        'schema_instruction': SCHEMA_INSTRUCTION,
        'prompt_template': PROMPT_TEMPLATE,
    }
    import hashlib

    def _short_file_info(path, hash_large=False):
        path = Path(path)
        if not path.exists():
            return f'{path} <missing>'
        stat = path.stat()
        digest = ''
        if hash_large or stat.st_size < 64 * 1024 * 1024:
            h = hashlib.md5()
            with path.open('rb') as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b''):
                    h.update(chunk)
            digest = f' md5={h.hexdigest()}'
        return f'{path} size={stat.st_size}{digest}'

    print('--- RKLLM HF eval provenance ---')
    print('MODEL_DIR =', MODEL_DIR)
    print('HF_EVAL_JSONL =', HF_EVAL_JSONL)
    print('HF_EVAL_JSONL info =', _short_file_info(HF_EVAL_JSONL, hash_large=True))
    for name in ['config.json', 'generation_config.json', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']:
        print(name, '=', _short_file_info(MODEL_DIR / name, hash_large=(name == 'model.safetensors')))
    try:
        with open(HF_EVAL_JSONL, 'r', encoding='utf-8') as f:
            first_eval = json.loads(next(line for line in f if line.strip()))
        print('HF_EVAL first input =', first_eval.get('input'))
        print('HF_EVAL first output =', first_eval.get('output'))
        print('HF_EVAL first prompt =')
        print(PROMPT_TEMPLATE.format(SCHEMA_INSTRUCTION, first_eval.get('input', ''), ''))
    except Exception as exc:
        print('HF_EVAL preview failed:', repr(exc))

    config_path.write_text(json.dumps(config, ensure_ascii=False), encoding='utf-8')
    script_path.write_text('\nimport gc\nimport json\nimport sys\nfrom pathlib import Path\n\nimport torch\nimport transformers\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig\n\nprint(\'[Clean native Transformers] python=\', sys.executable)\nprint(\'[Clean native Transformers] torch=\', torch.__version__)\nprint(\'[Clean native Transformers] transformers=\', transformers.__version__)\nprint(\'[Clean native Transformers] cuda_available=\', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print(\'[Clean native Transformers] cuda_device=\', torch.cuda.get_device_name(0))\ncfg = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\nmodel_dir = Path(cfg[\'model_dir\'])\njsonl_path = Path(cfg[\'jsonl\'])\nschema_instruction = cfg[\'schema_instruction\']\nprompt_template = cfg[\'prompt_template\']\nmax_rows = cfg[\'max_rows\']\nmax_new_tokens = cfg[\'max_new_tokens\']\n\ndef dtype_from_name(name):\n    if name == \'bfloat16\':\n        return torch.bfloat16\n    if name == \'float16\':\n        return torch.float16\n    if name == \'float32\':\n        return torch.float32\n    raise ValueError(f\'Unsupported dtype: {name}\')\n\ndef normalize_output(output):\n    if isinstance(output, dict):\n        return json.dumps(output, ensure_ascii=False, separators=(\',\', \':\'))\n    if isinstance(output, str):\n        try:\n            return json.dumps(json.loads(output), ensure_ascii=False, separators=(\',\', \':\'))\n        except json.JSONDecodeError:\n            return output.strip()\n    return str(output).strip()\n\ndef parse_json_maybe(text):\n    text = text.strip()\n    try:\n        return json.loads(text)\n    except Exception:\n        pass\n    start = text.find(\'{\')\n    if start < 0:\n        return None\n    depth = 0\n    in_str = False\n    esc = False\n    for i, ch in enumerate(text[start:], start):\n        if in_str:\n            if esc:\n                esc = False\n            elif ch == \'\\\\\':\n                esc = True\n            elif ch == \'"\':\n                in_str = False\n        else:\n            if ch == \'"\':\n                in_str = True\n            elif ch == \'{\':\n                depth += 1\n            elif ch == \'}\':\n                depth -= 1\n                if depth == 0:\n                    try:\n                        return json.loads(text[start:i + 1])\n                    except Exception:\n                        return None\n    return None\n\nrows = []\nwith jsonl_path.open(\'r\', encoding=\'utf-8\') as f:\n    for line in f:\n        if line.strip():\n            rows.append(json.loads(line))\n        if max_rows and len(rows) >= max_rows:\n            break\n\ndef build_prompt(input_text):\n    return prompt_template.format(schema_instruction, input_text, \'\')\n\ndef evaluate_dtype(dtype_name):\n    print(f\'\\n[Clean native Transformers] model={model_dir}\')\n    print(f\'[Clean native Transformers] dtype={dtype_name}\')\n    try:\n        tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True, fix_mistral_regex=True)\n    except TypeError:\n        tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    print(\'[Clean native Transformers] tokenizer_class=\', tokenizer.__class__.__name__)\n\n    model = AutoModelForCausalLM.from_pretrained(\n        str(model_dir),\n        torch_dtype=dtype_from_name(dtype_name),\n        device_map=\'cuda\',\n        trust_remote_code=True,\n    )\n    model.eval()\n    print(\'[Clean native Transformers] model_class=\', model.__class__.__name__)\n    print(\'[Clean native Transformers] config_torch_dtype=\', getattr(model.config, \'torch_dtype\', None))\n    print(\'[Clean native Transformers] loaded_param_dtype=\', next(model.parameters()).dtype)\n    model.generation_config = GenerationConfig.from_model_config(model.config)\n    model.generation_config.do_sample = False\n    model.generation_config.temperature = None\n    model.generation_config.top_p = None\n    model.generation_config.top_k = None\n    model.generation_config.pad_token_id = tokenizer.eos_token_id\n    model.generation_config.eos_token_id = tokenizer.eos_token_id\n\n    results = []\n    for idx, example in enumerate(rows, 1):\n        expected = json.loads(normalize_output(example[\'output\']))\n        prompt = build_prompt(example[\'input\'])\n        inputs = tokenizer([prompt], return_tensors=\'pt\').to(\'cuda\')\n        prompt_len = inputs.input_ids.shape[-1]\n        with torch.inference_mode():\n            outputs = model.generate(\n                **inputs,\n                max_length=prompt_len + max_new_tokens,\n                do_sample=False,\n                use_cache=True,\n                pad_token_id=tokenizer.eos_token_id,\n                eos_token_id=tokenizer.eos_token_id,\n            )\n        generated_text = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()\n        generated = parse_json_maybe(generated_text)\n        results.append({\n            \'input\': example[\'input\'],\n            \'expected\': expected,\n            \'generated_text\': generated_text,\n            \'generated\': generated,\n            \'json_valid\': generated is not None,\n            \'exact_match\': generated == expected,\n            \'intent_match\': generated is not None and generated.get(\'intent\') == expected.get(\'intent\'),\n            \'slots_match\': generated is not None and generated.get(\'slots\') == expected.get(\'slots\'),\n        })\n        if idx % 20 == 0 or idx == len(rows):\n            print(f\'[Clean native Transformers {dtype_name}] {idx}/{len(rows)} done\')\n\n    n = len(results)\n    summary = {\n        \'total\': n,\n        \'json_valid\': sum(r[\'json_valid\'] for r in results),\n        \'exact_match\': sum(r[\'exact_match\'] for r in results),\n        \'intent_match\': sum(r[\'intent_match\'] for r in results),\n        \'slots_match\': sum(r[\'slots_match\'] for r in results),\n    }\n    print(f\'--- Clean native Transformers {dtype_name} Evaluation Summary ---\')\n    for key, value in summary.items():\n        if key == \'total\':\n            print(f\'{key}: {value}\')\n        else:\n            print(f\'{key}: {value}/{n} = {value / n:.2%}\')\n\n    print(f\'\\n--- Clean native Transformers {dtype_name} Mismatches (first 10) ---\')\n    shown = 0\n    for idx, r in enumerate(results, 1):\n        if not r[\'exact_match\']:\n            print(f"#{idx} input: {r[\'input\']}")\n            print(\'expected:\', json.dumps(r[\'expected\'], ensure_ascii=False, separators=(\',\', \':\')))\n            print(\'generated_text:\', r[\'generated_text\'])\n            shown += 1\n            if shown >= 10:\n                break\n\n    del model, tokenizer\n    torch.cuda.empty_cache()\n    gc.collect()\n    return summary\n\nall_summaries = {}\nfor dtype_name in cfg[\'dtypes\']:\n    all_summaries[dtype_name] = evaluate_dtype(dtype_name)\n\nprint(\'\\n--- Clean native Transformers all summaries ---\')\nfor dtype_name, summary in all_summaries.items():\n    total = summary[\'total\'] or 1\n    print(\n        f"{dtype_name}: "\n        f"json_valid={summary[\'json_valid\']}/{total}={summary[\'json_valid\']/total:.2%}, "\n        f"exact={summary[\'exact_match\']}/{total}={summary[\'exact_match\']/total:.2%}, "\n        f"intent={summary[\'intent_match\']}/{total}={summary[\'intent_match\']/total:.2%}, "\n        f"slots={summary[\'slots_match\']}/{total}={summary[\'slots_match\']/total:.2%}"\n    )\n', encoding='utf-8')

    print('Running clean native Transformers eval before RKLLM conversion:')
    print('MODEL_DIR =', MODEL_DIR)
    print('HF_EVAL_JSONL =', HF_EVAL_JSONL)
    print('HF_EVAL_DTYPES =', HF_EVAL_DTYPES)
    print(' ', sys.executable, '-u', script_path, config_path)
    completed = subprocess.run(
        [sys.executable, '-u', str(script_path), str(config_path)],
        text=True,
        capture_output=True,
    )
    print('--- clean subprocess stdout ---')
    print(completed.stdout or '<empty>')
    print('--- clean subprocess stderr ---')
    print(completed.stderr or '<empty>')
    print('--- clean subprocess returncode ---')
    print(completed.returncode)
    if completed.returncode != 0:
        raise RuntimeError(f'Clean merged HF eval failed with exit code {completed.returncode}')


## 5. Convert with GPU and export RKLLM

The output filename includes the date, for example model_W8A8_RK3588_20260630.rkllm.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util
# Restore configuration after Colab runtime restart.
from pathlib import Path
from datetime import datetime

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

if 'MODEL_DIR' not in globals():
    MODEL_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output-mergeable-bf16/owon-qwen3-0.6b-merged-peft-safe-bf16')
if 'RKLLM_TOOLKIT_PACKAGE_DIR' not in globals():
    RKLLM_TOOLKIT_PACKAGE_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/rkllm-toolkit/packages')
if 'RKLLM_TOOLKIT_WHEEL' not in globals():
    RKLLM_TOOLKIT_WHEEL = None
if 'RKLLM_TOOLKIT_VERSION' not in globals():
    RKLLM_TOOLKIT_VERSION = '1.2.2'
if 'ALLOW_RKNN_LLM_CLONE' not in globals():
    ALLOW_RKNN_LLM_CLONE = False
if 'DATA_QUANT' not in globals():
    DATA_QUANT = MODEL_DIR / 'data_quant.json'
if 'JSONL_DATASET' not in globals():
    JSONL_DATASET = Path('/content/drive/MyDrive/qwen3-0.6B/LORA_train-qwen0.6B.jsonl')
if 'REGENERATE_DATA_QUANT' not in globals():
    REGENERATE_DATA_QUANT = True
if 'MAX_CALIBRATION_ROWS' not in globals():
    MAX_CALIBRATION_ROWS = None
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = MODEL_DIR
if 'OUTPUT_BASENAME' not in globals():
    OUTPUT_BASENAME = 'owon-qwen3-0.6b-mergeable-bf16-peft-safe'
if 'TARGET_PLATFORM' not in globals():
    TARGET_PLATFORM = 'RK3588'
if 'QUANTIZED_DTYPE' not in globals():
    QUANTIZED_DTYPE = 'W8A8'
if 'QUANTIZED_ALGORITHM' not in globals():
    QUANTIZED_ALGORITHM = 'normal'
if 'OPTIMIZATION_LEVEL' not in globals():
    OPTIMIZATION_LEVEL = 0
if 'NUM_NPU_CORE' not in globals():
    NUM_NPU_CORE = 3
if 'MAX_CONTEXT' not in globals():
    MAX_CONTEXT = 4096
if 'LOAD_DTYPE' not in globals():
    LOAD_DTYPE = 'float32'
if 'OUTPUT_PATH' not in globals():
    DATE_TAG = datetime.now().strftime('%Y%m%d')
    OUTPUT_PATH = OUTPUT_DIR / f'{OUTPUT_BASENAME}_{QUANTIZED_DTYPE}_{TARGET_PLATFORM}_{DATE_TAG}.rkllm'
print('MODEL_DIR =', MODEL_DIR)
print('DATA_QUANT =', DATA_QUANT)
print('OUTPUT_PATH =', OUTPUT_PATH)


os.environ['USE_TORCH'] = '1'
os.environ['USE_JAX'] = '0'
# Disable TensorFlow/Flax backends in transformers. RKLLM conversion only needs PyTorch.
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

RKLLM_RUNTIME_DEPS_NO_TORCH_TRANSFORMERS = [
    'colorlog==6.8.2',
    'tabulate==0.9.0',
    'jsonschema==4.23.0',
    'flatbuffers==24.3.25',
    'jsonlines==4.0.0',
    'sentencepiece==0.2.0',
    'protobuf<=4.25.4,>=4.21.6',
    'transformers_stream_generator==0.0.5',
    'einops==0.4.1',
    'tiktoken<=0.9.0,>=0.7.0',
    'datamodel_code_generator==0.26.0',
    'optimum==1.23.3',
    'timm==1.0.19',
]
RKLLM_RUNTIME_IMPORT_CHECKS = [
    'colorlog',
    'tabulate',
    'jsonschema',
    'flatbuffers',
    'jsonlines',
    'sentencepiece',
    'google.protobuf',
    'transformers_stream_generator',
    'einops',
    'tiktoken',
    'datamodel_code_generator',
    'optimum',
    'timm',
]

def missing_rkllm_runtime_modules():
    missing = []
    for module_name in RKLLM_RUNTIME_IMPORT_CHECKS:
        try:
            if importlib.util.find_spec(module_name) is None:
                missing.append(module_name)
        except ModuleNotFoundError:
            missing.append(module_name)
    return missing

missing_runtime_modules = missing_rkllm_runtime_modules()
if missing_runtime_modules:
    print('Missing RKLLM runtime modules before conversion:', missing_runtime_modules)
    print('Installing RKLLM runtime deps without changing torch/transformers/numpy...', flush=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', *RKLLM_RUNTIME_DEPS_NO_TORCH_TRANSFORMERS])
    missing_runtime_modules = missing_rkllm_runtime_modules()
    if missing_runtime_modules:
        raise ModuleNotFoundError(f'RKLLM runtime deps still missing after install: {missing_runtime_modules}')

from rkllm.api import RKLLM
if not DATA_QUANT.exists():
    quant_candidates = [
        MODEL_DIR / 'data_quant.json',
        Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output-mergeable-bf16/owon-qwen3-0.6b-merged-peft-safe-bf16/data_quant.json'),
    ]
    quant_candidates = [p for p in quant_candidates if p.exists()]
    if quant_candidates:
        DATA_QUANT = quant_candidates[0]
        print('Using discovered quant data:', DATA_QUANT)

for path in [MODEL_DIR, DATA_QUANT, OUTPUT_DIR]:
    if not Path(path).exists():
        raise FileNotFoundError(path)

llm = RKLLM()

print('Loading HuggingFace model on CUDA...')
ret = llm.load_huggingface(
    model=str(MODEL_DIR),
    model_lora=None,
    device='cuda',
    dtype=LOAD_DTYPE,
    custom_config=None,
    load_weight=True,
)
if ret != 0:
    raise RuntimeError(f'Load model failed: {ret}')

print('Building RKLLM...')
ret = llm.build(
    do_quantization=True,
    optimization_level=OPTIMIZATION_LEVEL,
    quantized_dtype=QUANTIZED_DTYPE,
    quantized_algorithm=QUANTIZED_ALGORITHM,
    target_platform=TARGET_PLATFORM,
    num_npu_core=NUM_NPU_CORE,
    extra_qparams=None,
    dataset=str(DATA_QUANT),
    hybrid_rate=0,
    max_context=MAX_CONTEXT,
)
if ret != 0:
    raise RuntimeError(f'Build model failed: {ret}')

print('Exporting:', OUTPUT_PATH)
ret = llm.export_rkllm(str(OUTPUT_PATH))
if ret != 0:
    raise RuntimeError(f'Export model failed: {ret}')

print('Conversion succeeded:', OUTPUT_PATH)
print('File size MB:', OUTPUT_PATH.stat().st_size / 1024 / 1024)



## 6. Optional local download

If OUTPUT_PATH points to /content instead of Drive, uncomment this cell to download the file.


In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))
